In [1]:
import pandas as pd
import re

In [2]:
artistless = pd.read_csv("data/features-annotated-artistless.csv")
non_featuring = artistless[artistless["Featuring"] == 0]
featuring = artistless[artistless["Featuring"] == 1]

In [3]:
# Take a string with featuring terms and produce a comma separated list of featured artists
def strip_feature(artist_str):
    # Check for "with <artists1> feat/ft. <artists2>" and "with <artists1> & feat/ft. <artists2>"
    # Handle "with <artists1> & feat/ft. <artists2>" and "with <artists1> feat/ft. <artists2>"
    s = artist_str
    def join_groups(*groups):
        return ','.join(g.strip() for g in groups if g)

    s = re.sub(
        r'with\s+([^\(\)\[\]]+?)\s*&\s*(?:feat|ft)\.?\s*([^\(\)\[\]]+)',
        lambda m: join_groups(m.group(1), m.group(2)),
        s,
        flags=re.IGNORECASE
    )
    s = re.sub(
        r'with\s+([^\(\)\[\]]+?)\s*(?:feat|ft)\.?\s*([^\(\)\[\]]+)',
        lambda m: join_groups(m.group(1), m.group(2)),
        s,
        flags=re.IGNORECASE
    )
    # Handle "<artists1> & <artists2> feat/ft. <artists3>"
    s = re.sub(
        r'([^\(\)\[\]]+?)\s*&\s*([^\(\)\[\]]+?)\s*(?:feat|ft)\.?\s*([^\(\)\[\]]+)',
        lambda m: join_groups(m.group(1), m.group(2), m.group(3)),
        s,
        flags=re.IGNORECASE
    )
    # Handle "<artists1> feat/ft. <artists2>"
    s = re.sub(
        r'([^\(\)\[\]]+?)\s*(?:feat|ft)\.?\s*([^\(\)\[\]]+)',
        lambda m: join_groups(m.group(1), m.group(2)),
        s,
        flags=re.IGNORECASE
    )

    # Remove "duet with <artist>"
    s = re.sub(r'duet with\s*[^\(\)\[\],]+', '', s, flags=re.IGNORECASE)

    # Remove all featuring terms in data-set 
    s = re.sub(r'\b(feat\.?\s*|featuring\s*|starring\s*|with\s*|con\s*|ft\.?\s*| remix edit\s*|w/\s*)\b\.?', '', s, flags=re.IGNORECASE)
    # Replace splitters with comma
    s = re.sub(r'\s*&\s*', ',', s) # <artist1> & <artist2>
    s = re.sub(r'\s* and \s*', ',', s) # <artist 1> and <artist2>
    s = re.sub(r'(?<!\,)\s+x\s+', ',', s) # <artist 1> x <artist2>, but not ", x ambassadors"
    s = re.sub(r'\s*, \s*', ',', s) # <artist1>, <artist2>
    s = re.sub(r'\s*/\s*', ',', s) # <artist1> / <artist2>
    s = re.sub(r'\s*vs\.\s*', ',', s, flags=re.IGNORECASE) # <artist1> vs. <artist2>
    return s.strip()

# Remove additional information about artists, e.g. "of/from <band>"
# Many of these are for individual cases
def parse_artist(artist):
    artist = re.sub(r'(of|from)\s+[^,]+', '', artist, flags=re.IGNORECASE) # <artist> of/from <band>
    artist = re.sub(r'his orchestra', '', artist, flags=re.IGNORECASE) # <artist> and his orchestra
    artist = re.sub(r'\'s orchestra', '', artist, flags=re.IGNORECASE) # <artist>'s orchestra
    artist = re.sub(r'chorus', '', artist, flags=re.IGNORECASE) # <artist> and chorus
    artist = re.sub(r'present\s+[^,]+', '', artist, flags=re.IGNORECASE) # <artists> present <show>
    artist = re.sub(r'as the people', '', artist, flags=re.IGNORECASE) # <artist> as the people
    artist = re.sub(r'remix;|remix', '', artist, flags=re.IGNORECASE) # <artist> remix or remix;
    artist = re.sub(r' radio edit', '', artist, flags=re.IGNORECASE) # <artist> radio edit
    artist = re.sub(r'rration by ', '', artist, flags=re.IGNORECASE) # rration by <artist>, note that "na" is absent from the dataset due to terrible cleaning
    artist = re.sub(r'\.*$', '', artist.strip())
    return artist

featured_artists = set()

featured_artists_title = {}
for title in featuring["Title"]:
    # Remove "." from the end of titles (helps with some featuring)
    title = title.rstrip(".")

    # Check for bracketed strings, e.g. (feat. sza) or [with future & chris brown]
    if (re.search(r"\(.*\)", title) or re.search(r"\[.*\]", title)):
        # Songs may have multiple bracket strings, e.g. superhero (heroes & villains) [with future & chris brown]
        brackets = re.findall(r"(\(.*?\)|\[.*?\])", title)

        # For each found bracket we parse to find artists
        for bracket in brackets:
            bracket = bracket.strip("()[]")
            stripped = strip_feature(bracket)

            # If we've changed something then we've successfully matched something in the regular expression and will have an artist list
            if stripped != bracket:
                featured = list(filter(lambda x: x != "", map(parse_artist, stripped.split(","))))
    
    # Non-bracketed features almost always come after a -
    elif len(split := title.split(" - ")) >= 2:
            stripped = strip_feature(split[-1])
            featured = list(filter(lambda x: x != "", map(parse_artist, stripped.split(","))))

    # Those which do not come after either feat. or ft.
    elif " feat. " in title:
        featured = title.split("feat. ")[1].split(", ")
    elif " ft. " in title:
        featured = title.split("ft. ")[1].split(", ")

    # Extend the featured artists set with the newly found featured artists
    featured_artists = featured_artists | set(featured)
    featured_artists_title[title] = featured
    

In [4]:
# Split the 'Artists' column by comma and flatten the list
all_artists = set(artistless['Artists'].str.split(',').explode().str.strip())
for artist in (featured_artists - all_artists):
    print(artist)

x ambassadors
conor mayrd
guilherme
fabolous
vula
reason
koba lad
u-god
peven everett
lagola
cade
ashanti
mara sattei
drefgold
kim jong wan
lennox
geolier
marcus mumford
snøw
mufasa
nyla
mae muller
m.i.a
the b. swanson quartet
adam levine
220 kid
phresher
chris jeday
daryl's version
slowthai
mc rebecca
raekwon
olivia o'brien
wyclef jean
ink
ray blk
quinn xcii
ybn cordae
zebra katz
nstasia
jehnny beth
pretty sister
rupaul
ozu
lenny santos
prod
gringo
jt
billy idol
vargas
rza
amber coffman
jake gosling
ape drums
dan caplen
big rube
s-x
nechie
oumou sangaré
the puppini sisters
bono
esperanza spalding
rah swish
towkio
henning may
jesse
sosa geek
darkoo
mack maine
sjava
rei brown
judah
white gold
theresa rex
kellin quinn
j. davi$
kxng crooked
ilovemakonnen
yxng bane
scorey
princess nokia
the rattpack
norma jean martine
jay z
alu
luvli
emilee
jace
e1 (3x3)
ky-mani marley
dan reynolds
aj tracey
boyz ii men
mitchell ayres
shania twain
nick waterhouse
clams casino
regie carter
yugen blakrok
mis